In [ ]:
# Changing column names adjusted for Snowflake import
# Preparing data for Snowflake entry
# Creating CSV for Snowflake import
import numpy as np
import pandas as pd 

In [109]:

def clean_string(string, remove_list, format_to_int: bool = False, format_to_float: bool = False, col_name: str = ''):
    # Check if the string is np.nan
    if isinstance(string, float) and np.isnan(string):
        return np.nan
    
    if string is None:
        return np.nan 
 
    # Remove all strings from the list starting with the given string 
    for item in remove_list:
        string = string.replace(item, "")
    
    # If the value is true, convert it to an integer
    if format_to_int:
        return to_int(string, col_name)
    
    # If the value is true, convert it to a float
    if format_to_float:
        return to_float(string, col_name)
     
    return string

def to_int(string, col_name: str = ''):
    # Check if the remaining string can be converted to an integer 
    try: 
        return int(string)
    except ValueError:
        print(f'To Int Error: {string}, Col Name: {col_name}')
        return np.nan
    
def to_float(string, col_name: str = ''): 
    # Check if the remaining string can be converted to a float
    try: 
        return float(string)
    except ValueError:
        print(f'To Float Error: {string}, Col Name: {col_name}')
        return np.nan
    
# Function to convert "Size" to integer (or float)
def convert_size(size):
    if isinstance(size, str):  # Check if the value is of type string
        if 'k' in size:
            # If 'k' is in the size, multiply by 1000
            return float(size.replace('k', '')) * 1000
        elif 'm' in size:
            # If 'm' is in the size, multiply by 1,000,000
            return float(size.replace('m', '')) * 1000000
        elif 'b' in size:
            return float(size.replace('b', '')) * 1000000000
        else:
            # If it is not 'k' or 'm', simply convert to int (or float)
            return float(size)
    else:
        # If it is not a string (e.g., already a number), return the value as is
        return size

In [110]:
columns_to_load = ["Address", "About", "Year Built", "Sold On", "Tax Amount", "Owner", "Contact Name", "Contact Relationship", "Phone Number", "Email", "Addresses", "Link"]

# Load Data
df = pd.read_excel(f'MLSA_Reonomy.xlsx', usecols=columns_to_load)

# Drop rows with all NaN values
df.dropna(axis=0, how="all", inplace=True)

df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ', '_')

In [111]:
# Split Address into multiple columns
df[['property_street', 'property_city', 'property_state_zip']] = df['address'].str.split(',', expand=True) 
df['property_state_zip'] = df['property_state_zip'].str.strip()
df[['property_state', 'property_zip_code']] = df['property_state_zip'].str.split(' ', expand=True)

# Split Addresses into multiple columns
df[['street', 'city', 'state_zip']] = df['addresses'].str.split(',', expand=True) 
df['state_zip'] = df['state_zip'].str.strip()
df[['state', 'zip_code']] = df['state_zip'].str.split(' ', expand=True)

# Split About into two columns 
df[['size_sf', 'type']] = df['about'].str.split(' SF ', expand=True) 
df['size_sf'] = df['size_sf'].apply(convert_size)

# Split Sold On into month, year, and amount 
df[['sold_date', 'sold_price']] = df['sold_on'].str.split(' for ', expand=True)
df[['sold_month', 'sold_year']] = df['sold_date'].str.split(' ', expand=True)
df['sold_price'] = df['sold_price'].str.replace('$', '')
df['sold_price'] = df['sold_price'].apply(convert_size)

# Tax amount
df['tax_amount'] = df['tax_amount'].apply(convert_size)

# Owner
df['owner'] = df['owner'].str.title()

# Phone
df['phone_number'] = df['phone_number'].apply(lambda x: clean_string(x, ['-'], format_to_int=True, col_name='phone_number')) 
df['phone_number'] = df['phone_number'].astype(pd.Int64Dtype())

# Ensure it remains an int
for col in ['property_zip_code', 'zip_code', 'year_built', 'sold_year']:
    df[col] = df[col].apply(lambda x: clean_string(x, [], format_to_int=True, col_name=col))
    df[col] = df[col].astype(pd.Int64Dtype())
    
# Ensure it remains a float
for col in ['size_sf', 'sold_price', 'tax_amount']:
    df[col] = df[col].apply(lambda x: clean_string(x, [], format_to_float=True, col_name=col))
    
# Scrape Date
df['scrape_date'] = pd.to_datetime('today').normalize()

df = df.drop(columns=['state_zip', 'property_state_zip', 'address', 'about', 'sold_on', 'sold_date', 'addresses'])

output_columns_order = ['property_street', 'property_city', 'property_state', 'property_zip_code', 'size_sf', 'type',
                        'year_built', 'sold_month', 'sold_year', 'sold_price', 'tax_amount', 'owner', 'contact_name',
                        'contact_relationship', 'phone_number', 'email', 'street', 'city', 'state', 'zip_code', 'link',
                        'scrape_date']

df = df[output_columns_order]

df.to_csv('mlsa_reonomy_output.csv', index=False)

In [112]:
print(df['phone_number'].count())
print(df['phone_number'].dropna().head(100))

7866
0       8176458836
1       8176458836
2       2814409642
3      19036636503
4       8172468777
          ...     
109     3617299155
110     4169884729
111    19178176613
112     2102735300
113    12105345799
Name: phone_number, Length: 100, dtype: Int64
